# KMC baseline simulator v5.1

**Public-release note.** This notebook is part of a GitHub-ready version of the KMC memristor project. Notebook outputs were stripped to keep the repository lightweight, and path settings were adjusted to use repository-relative locations where needed.

**Purpose:** Baseline 2D KMC memristor / filament simulator. Includes the core lattice model, event engine, morphology metrics, and diagnostic plots.

**Main manuscript role:** Core simulation engine used as the starting point for all later phase-specific notebooks.

**Default assumption:** run the notebook from inside this repository so that the `results/` directory can be discovered automatically.


In [ ]:
# -*- coding: utf-8 -*-
"""
KMC RRAM / ECM 2D filament simulator — v5.1 (Feb 2026)

你提到的几个问题，这个版本逐个修正/补齐：
1) 稳定性分析（convergence study）没被取消：RUN_CONVERGENCE=True 会画图+表格；而你截图里 x 轴是 N_seeds。
2) E–T 相图：默认 x=T(K), y=E(V/nm)。温度不是“纵坐标”，所以不叫“温度纵坐标错”。
3) Boxplot 坐标变 0..12/0..15：这是 seaborn 把类别当 index；已强制 categorical + 指定 ticklabels。
4) HRS / 临界 / LRS / Reset-HRS 的空位(ION)分布和丝几何：DEMO 输出四态 2D states + ION map + bottom-connected FIL mask + 3D revolve + 形貌表。
5) 图更好看：统一字号/布局/轴、取消 offset、收敛图与敏感性图更清晰。
6) 收敛分析支持多温度：CONV_T_LIST 默认 [400,700,900]。

用法：
- 命令行：python kmc_rram_v5_1.py
- Jupyter：import kmc_rram_v5_1 as kmc; kmc.main()

注意：
- 如果 DEVICE_SEEDS_SCAN 只有一个（例如 [2025]），扫描等价于“固定一块材料无序样本”，会出现系统性偏边的器件实例。
  想做“材料无序平均”，把 DEVICE_SEEDS_SCAN 改成 [2025,2026,2027]（会慢但更稳）。
"""

import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from numba import njit
from joblib import Parallel, delayed
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.ticker import ScalarFormatter
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

sns.set_theme(style="whitegrid", context="paper", font_scale=1.10)

# =========================
# 0) 主要可调参数（集中在这）
# =========================
FAST_MODE = False

# Geometry (cells)
WIDTH_CELLS = 24
THICKNESS_CELLS = 36   # z=0 bottom electrode, z=H-1 top electrode
TOX_NM = 100.0
XSPAN_NM = 70.0

Z_BOT = 0
Z_TOP = THICKNESS_CELLS - 1
Z_OX_START = 1
Z_OX_END = THICKNESS_CELLS - 2

CELL_Z_NM = TOX_NM / (THICKNESS_CELLS - 2)
CELL_X_NM = XSPAN_NM / (WIDTH_CELLS - 1)

# physics constants
KB = 8.617e-5  # eV/K
NU0 = 1e12     # 1/s attempt

# States
S_EMPTY = np.int8(0)
S_ION   = np.int8(1)  # vacancy/ion carrier (你的模型里当“空位”也完全OK)
S_FIL   = np.int8(2)  # conductive filament phase
S_EBOT  = np.int8(3)
S_ETOP  = np.int8(4)

# barrier map
BASE_BARRIER_EV = 0.85
RANDOM_SIGMA_EV = 0.12
MIN_BARRIER_EV  = 0.08

# injection (proxy)
PHIB_BOTTOM = 0.70
PHIB_TOP    = 0.90
E_INJECT0_EV = 0.35

# field coupling
ALPHA_FIELD = 0.22
FIELD_ENHANCE_MIG  = 10.0
FIELD_ENHANCE_GROW = 120.0

ENABLE_TIP_FOCUS  = True
TIP_FOCUS_POWER   = 1.8
TIP_FOCUS_MAX_VNM = 10.0

# redox/growth
E_GROW0_EV   = 1.12
GAMMA_GROW   = 0.020
DELTA_ATTACH = 0.24

# hop slow near tip
HOP_SLOW_NEAR_TIP = 0.05
TIP_BEHIND = 0
TIP_AHEAD  = 5

# reset (dissolution)
ENABLE_RESET = True
E_DISS0_EV   = 1.35
GAMMA_DISS   = 0.015

# filament morphology controls
FOCUS_KAPPA = 0.05
E_LATERAL_NOISE_VNM = 0.01
GROW_PREF_FORWARD = 2.0
GROW_PENALTY_LATERAL = 0.35

# nucleation
NUCLEATION_MODE = "centered_random_multi"  # "center" | "random" | "random_multi" | "centered_random_multi"
NUC_SEEDS = 3

# post-connect thickening (define LRS vs critical)
ENABLE_POST_THICKEN = True
POST_CONNECT_STEPS = 4000

# center bias (仅作为“电极形貌/场聚焦”的可视化 proxy；要完全对称则关掉)
ENABLE_CENTER_BIAS = True
CENTER_BIAS_STRENGTH = 0.30
CENTER_BIAS_SIGMA_CELLS = 4.0

# KMC controls
N_INIT_IONS = 140
INIT_ION_Z_MAX_FRAC = 0.35
MAX_STEPS_SET   = 120000
MAX_STEPS_RESET = 100000
STOP_IF_CONNECTED_SET = True
STOP_IF_DISCONNECTED_RESET = True

RATE_SAMPLE_MAX = 1200
RATE_SAMPLE_STRIDE = 7
MASK_UPDATE_STRIDE = 5

# Scan grids
E_VALS_FULL = np.linspace(0.010, 0.16, 16)
T_VALS_FULL = np.linspace(300, 900, 13)
N_SEEDS_FULL = 20

E_VALS_FAST = np.linspace(0.02, 0.14, 7)
T_VALS_FAST = np.linspace(300, 900, 7)
N_SEEDS_FAST = 5

E_VALS = E_VALS_FAST if FAST_MODE else E_VALS_FULL
T_VALS = T_VALS_FAST if FAST_MODE else T_VALS_FULL
N_SEEDS_PER_POINT = N_SEEDS_FAST if FAST_MODE else N_SEEDS_FULL

P_FORM_MIN   = 0.25
P_SWITCH_MIN = 0.25

T_REF_MORPH = 700.0
DRIFT_THRESH_NM = 8.0

# parallel
N_JOBS = 12
PARALLEL_PREFER = "threads"
CHUNK_SIZE = 200
JOBLIB_VERBOSE = 0

# DEMO
RUN_DEMO = True
E_DEMO = 0.080
T_DEMO = 700.0

BARRIER_MODE_DEMO = "random"   # "random" | "layered" | "aniso"
BARRIER_MODE_SCAN = "random"

DEVICE_SEEDS_SCAN = [2025]     # 建议多几个做无序平均：[2025,2026,2027]

# store heavy
STORE_RATE_SAMPLES_DEMO = True
STORE_RATE_SAMPLES_SCAN = False

# analysis
RUN_SCAN = True
RUN_CONVERGENCE = True
RUN_SENSITIVITY = True

# convergence
CONV_E0 = 0.080
CONV_T_LIST = [400.0, 700.0, 900.0]
CONV_N_LIST = [3, 5, 8, 12, 20, 30, 40, 60]

# sensitivity (at one op point)
SENS_E0 = 0.080
SENS_T0 = 700.0
SENS_NSEEDS = 16
SENS_PERT_FRAC = 0.15


# =========================
# 1) utilities / plotting
# =========================
def _no_offset(ax):
    fmt = ScalarFormatter(useOffset=False)
    fmt.set_scientific(False)
    ax.xaxis.set_major_formatter(fmt)
    ax.yaxis.set_major_formatter(fmt)

@njit(cache=True)
def _clamp(Eb, Eb_min):
    return Eb_min if Eb < Eb_min else Eb

@njit(cache=True)
def _tip_field(E_base, gap_cells, power, emax, enable):
    if not enable:
        return E_base
    g = gap_cells
    if g < 1:
        g = 1
    val = E_base * (1.0 + 1.0 / g) ** power
    return emax if val > emax else val

@njit(cache=True)
def _center_bias_factor(x, w, enable, strength, sigma_cells):
    if not enable:
        return 1.0
    xc = (w - 1) / 2.0
    dx = x - xc
    s2 = sigma_cells * sigma_cells
    if s2 < 1e-9:
        return 1.0
    return 1.0 + strength * np.exp(-0.5 * dx * dx / s2)


# =========================
# 2) topology (connected mask / tip)
# =========================
@njit(cache=True)
def _bfs_bottom_connected_mask(occ):
    w, h = occ.shape
    mask = np.zeros((w, h), dtype=np.uint8)
    qx = np.empty(w*h, dtype=np.int32)
    qz = np.empty(w*h, dtype=np.int32)
    head = 0
    tail = 0
    # bottom electrode line as sources
    for x in range(w):
        mask[x, 0] = 1
        qx[tail] = x; qz[tail] = 0
        tail += 1
    # BFS
    while head < tail:
        x = qx[head]; z = qz[head]; head += 1
        # 4-neighbor expansion only into FIL or electrodes
        if x > 0 and mask[x-1, z] == 0:
            s = occ[x-1, z]
            if s == S_FIL or s == S_EBOT or s == S_ETOP:
                mask[x-1, z] = 1; qx[tail] = x-1; qz[tail] = z; tail += 1
        if x < w-1 and mask[x+1, z] == 0:
            s = occ[x+1, z]
            if s == S_FIL or s == S_EBOT or s == S_ETOP:
                mask[x+1, z] = 1; qx[tail] = x+1; qz[tail] = z; tail += 1
        if z > 0 and mask[x, z-1] == 0:
            s = occ[x, z-1]
            if s == S_FIL or s == S_EBOT or s == S_ETOP:
                mask[x, z-1] = 1; qx[tail] = x; qz[tail] = z-1; tail += 1
        if z < h-1 and mask[x, z+1] == 0:
            s = occ[x, z+1]
            if s == S_FIL or s == S_EBOT or s == S_ETOP:
                mask[x, z+1] = 1; qx[tail] = x; qz[tail] = z+1; tail += 1
    return mask

@njit(cache=True)
def _connected_to_top(occ, mask):
    w, h = occ.shape
    z = h - 2
    for x in range(w):
        if mask[x, z] == 1 and occ[x, z+1] == S_ETOP:
            return True
    return False

@njit(cache=True)
def _tip_stats_from_filament(occ, mask):
    w, h = occ.shape
    tip_z = 1
    found_any = False
    for z in range(h-2, 0, -1):
        found = False
        for x in range(w):
            if mask[x, z] == 1 and occ[x, z] == S_FIL:
                tip_z = z
                found = True
                found_any = True
                break
        if found:
            break
    if not found_any:
        return 1, (w-1)/2.0
    sx = 0.0
    cnt = 0.0
    for x in range(w):
        if mask[x, tip_z] == 1 and occ[x, tip_z] == S_FIL:
            sx += x; cnt += 1.0
    tip_x = sx / cnt if cnt > 0 else (w-1)/2.0
    return tip_z, tip_x


# =========================
# 3) local neighborhood helpers
# =========================
@njit(cache=True)
def _count_attach_neighbors_including_electrodes(occ, x, z):
    cnt = 0
    w, h = occ.shape
    if x > 0:
        s = occ[x-1, z]
        if s == S_FIL or s == S_EBOT or s == S_ETOP: cnt += 1
    if x < w-1:
        s = occ[x+1, z]
        if s == S_FIL or s == S_EBOT or s == S_ETOP: cnt += 1
    if z > 0:
        s = occ[x, z-1]
        if s == S_FIL or s == S_EBOT or s == S_ETOP: cnt += 1
    if z < h-1:
        s = occ[x, z+1]
        if s == S_FIL or s == S_EBOT or s == S_ETOP: cnt += 1
    return cnt

@njit(cache=True)
def _count_fil_neighbors_only(occ, x, z):
    cnt = 0
    w, h = occ.shape
    if x > 0   and occ[x-1, z] == S_FIL: cnt += 1
    if x < w-1 and occ[x+1, z] == S_FIL: cnt += 1
    if z > 0   and occ[x, z-1] == S_FIL: cnt += 1
    if z < h-1 and occ[x, z+1] == S_FIL: cnt += 1
    return cnt

@njit(cache=True)
def _neighbor_flags_fil(occ, x, z):
    w, h = occ.shape
    left = (x > 0 and occ[x-1, z] == S_FIL)
    right = (x < w-1 and occ[x+1, z] == S_FIL)
    down = (z > 0 and occ[x, z-1] == S_FIL)
    up = (z < h-1 and occ[x, z+1] == S_FIL)
    return left, right, down, up


# =========================
# 4) morphology metrics (meander/neck/branches/drift/tortuosity)
# =========================
@njit(cache=True)
def _morphology_metrics_filament(occ, mask, cell_x_nm, drift_thresh_nm):
    w, h = occ.shape
    # neck
    neck_cells = 10**9
    for z in range(1, h-1):
        c = 0
        for x in range(w):
            if mask[x, z] == 1 and occ[x, z] == S_FIL:
                c += 1
        if c > 0 and c < neck_cells:
            neck_cells = c
    if neck_cells == 10**9:
        neck_cells = 0
    neck_nm = neck_cells * cell_x_nm

    # branches (deg>=3)
    branches = 0
    for z in range(1, h-1):
        for x in range(w):
            if mask[x, z] == 1 and occ[x, z] == S_FIL:
                deg = 0
                if x > 0   and mask[x-1, z] == 1 and occ[x-1, z] == S_FIL: deg += 1
                if x < w-1 and mask[x+1, z] == 1 and occ[x+1, z] == S_FIL: deg += 1
                if z > 1   and mask[x, z-1] == 1 and occ[x, z-1] == S_FIL: deg += 1
                if z < h-2 and mask[x, z+1] == 1 and occ[x, z+1] == S_FIL: deg += 1
                if deg >= 3:
                    branches += 1

    # COMx drift
    sx = 0.0; ctot = 0.0
    for z in range(1, h-1):
        for x in range(w):
            if mask[x, z] == 1 and occ[x, z] == S_FIL:
                sx += x; ctot += 1.0
    comx_nm = (sx / ctot) * cell_x_nm if ctot > 0 else -1.0
    center_nm = ((w-1)/2.0) * cell_x_nm
    drift_flag = 0
    if comx_nm >= 0.0 and abs(comx_nm - center_nm) > drift_thresh_nm:
        drift_flag = 1

    # meander RMS + tortuosity (sum |dx| between adjacent occupied layers)
    sum_x = 0.0; sum_x2 = 0.0; nz = 0.0
    tort = 0.0; last_x = -1.0
    for z in range(1, h-1):
        szz = 0.0; cntz = 0.0
        for x in range(w):
            if mask[x, z] == 1 and occ[x, z] == S_FIL:
                szz += x; cntz += 1.0
        if cntz > 0:
            xz = szz / cntz
            sum_x += xz; sum_x2 += xz * xz; nz += 1.0
            if last_x >= 0.0:
                tort += abs(xz - last_x) * cell_x_nm
            last_x = xz
    if nz > 1.0:
        mean_x = sum_x / nz
        var_x = sum_x2 / nz - mean_x * mean_x
        if var_x < 0.0: var_x = 0.0
        meander_rms_nm = np.sqrt(var_x) * cell_x_nm
    else:
        meander_rms_nm = 0.0
    return meander_rms_nm, neck_nm, branches, comx_nm, drift_flag, tort


# =========================
# 5) barrier map generators
# =========================
def make_barrier_map_random(seed=0, base=BASE_BARRIER_EV, sigma=RANDOM_SIGMA_EV):
    rng = np.random.default_rng(seed)
    em = base + rng.normal(0.0, sigma, size=(WIDTH_CELLS, THICKNESS_CELLS))
    em[:, Z_BOT] = base
    em[:, Z_TOP] = base
    em = np.maximum(em, MIN_BARRIER_EV)
    return em.astype(np.float64)

def make_barrier_map_layered(z_regions, Eb_values, sigma=RANDOM_SIGMA_EV, seed=0):
    assert len(z_regions) == len(Eb_values)
    rng = np.random.default_rng(seed)
    em = np.full((WIDTH_CELLS, THICKNESS_CELLS), BASE_BARRIER_EV, dtype=np.float64)
    for (z0, z1), Eb in zip(z_regions, Eb_values):
        z0 = max(0, int(z0)); z1 = min(THICKNESS_CELLS-1, int(z1))
        em[:, z0:z1+1] = float(Eb)
    em += rng.normal(0.0, sigma, size=em.shape)
    em[:, Z_BOT] = BASE_BARRIER_EV
    em[:, Z_TOP] = BASE_BARRIER_EV
    em = np.maximum(em, MIN_BARRIER_EV)
    return em

def make_barrier_maps_aniso(Eb_x, Eb_z, sigma=RANDOM_SIGMA_EV, seed=0):
    rng = np.random.default_rng(seed)
    em_x = Eb_x + rng.normal(0.0, sigma, size=(WIDTH_CELLS, THICKNESS_CELLS))
    em_z = Eb_z + rng.normal(0.0, sigma, size=(WIDTH_CELLS, THICKNESS_CELLS))
    for em in (em_x, em_z):
        em[:, Z_BOT] = BASE_BARRIER_EV
        em[:, Z_TOP] = BASE_BARRIER_EV
    em_x = np.maximum(em_x, MIN_BARRIER_EV).astype(np.float64)
    em_z = np.maximum(em_z, MIN_BARRIER_EV).astype(np.float64)
    return em_x, em_z


# =========================
# 6) init occupancy
# =========================
def init_occ(seed=0):
    rng = np.random.default_rng(seed)
    occ = np.zeros((WIDTH_CELLS, THICKNESS_CELLS), dtype=np.int8)
    occ[:, Z_BOT] = S_EBOT
    occ[:, Z_TOP] = S_ETOP

    zmax = int(max(2, Z_OX_END * INIT_ION_Z_MAX_FRAC))
    cnt = 0
    while cnt < N_INIT_IONS:
        x = int(rng.integers(0, WIDTH_CELLS))
        z = int(rng.integers(Z_OX_START, zmax + 1))
        if occ[x, z] == S_EMPTY:
            occ[x, z] = S_ION
            cnt += 1

    if NUCLEATION_MODE == "center":
        occ[WIDTH_CELLS//2, 1] = S_FIL
    elif NUCLEATION_MODE == "random":
        occ[int(rng.integers(0, WIDTH_CELLS)), 1] = S_FIL
    elif NUCLEATION_MODE == "random_multi":
        xs = rng.choice(np.arange(WIDTH_CELLS), size=min(NUC_SEEDS, WIDTH_CELLS), replace=False)
        for x in xs: occ[int(x), 1] = S_FIL
    else:
        xc = (WIDTH_CELLS - 1) / 2.0
        xs = rng.choice(np.arange(WIDTH_CELLS), size=min(NUC_SEEDS, WIDTH_CELLS), replace=False)
        xs = sorted(xs, key=lambda x: abs(x - xc))
        for x in xs: occ[int(x), 1] = S_FIL

    return occ


# =========================
# 7) KMC core (anisotropic barriers em_x/em_z)
# =========================
@njit(cache=True)
def _run_kmc(
    occ0, em_x, em_z,
    E_avg_vnm, T,
    max_steps,
    do_growth, do_dissolve,
    stop_if_connected,
    stop_if_disconnected,
    seed,
    rate_sample_max,
    rate_sample_stride,
    mask_update_stride,
    alpha_field,
    min_barrier,
    field_enh_mig,
    field_enh_grow,
    tip_focus_power,
    tip_focus_max,
    enable_tip_focus,
    e_grow0,
    gamma_grow,
    delta_attach,
    hop_slow_near_tip,
    tip_behind,
    tip_ahead,
    e_diss0,
    gamma_diss,
    phib_bottom,
    phib_top,
    e_inject0,
    focus_kappa,
    e_lat_noise_vnm,
    grow_pref_forward,
    grow_penalty_lateral,
    cell_z_nm,
    enable_center_bias,
    center_bias_strength,
    center_bias_sigma_cells
):
    np.random.seed(seed)
    occ = occ0.copy()
    w, h = occ.shape

    t = 0.0
    t_connect = np.nan
    last_diss_x = -1
    last_diss_z = -1

    n_hopx = 0
    n_hopz = 0
    n_grow = 0
    n_diss = 0
    n_inj  = 0

    rates_s = np.zeros(rate_sample_max, dtype=np.float64)
    ns = 0

    max_events = w * (h-2) * 7 + w * 2
    rates = np.empty(max_events, dtype=np.float64)
    etype = np.empty(max_events, dtype=np.int8)     # 0 hop, 1 grow, 2 diss, 3 injB, 4 injT
    ex = np.empty(max_events, dtype=np.int16)
    ez = np.empty(max_events, dtype=np.int16)
    enx = np.empty(max_events, dtype=np.int16)
    enz = np.empty(max_events, dtype=np.int16)

    mask = _bfs_bottom_connected_mask(occ)
    connected = _connected_to_top(occ, mask)
    tip_z, tip_x = _tip_stats_from_filament(occ, mask)

    for step in range(max_steps):

        if step % mask_update_stride == 0:
            mask = _bfs_bottom_connected_mask(occ)
            connected = _connected_to_top(occ, mask)
            tip_z, tip_x = _tip_stats_from_filament(occ, mask)

            if connected and np.isnan(t_connect):
                t_connect = t
                if stop_if_connected:
                    energy_proxy_ev = abs(E_avg_vnm) * cell_z_nm * n_hopz
                    return (occ, t_connect, True, rates_s, ns, last_diss_x, last_diss_z,
                            n_hopx, n_hopz, n_grow, n_diss, n_inj, energy_proxy_ev)

            if (not connected) and (not np.isnan(t_connect)) and stop_if_disconnected:
                energy_proxy_ev = abs(E_avg_vnm) * cell_z_nm * n_hopz
                return (occ, t_connect, False, rates_s, ns, last_diss_x, last_diss_z,
                        n_hopx, n_hopz, n_grow, n_diss, n_inj, energy_proxy_ev)

        gap = (h - 1) - tip_z
        E_mig = abs(E_avg_vnm) * field_enh_mig
        E_grow_base = abs(E_avg_vnm) * field_enh_grow
        E_tip = _tip_field(E_grow_base, gap, tip_focus_power, tip_focus_max, enable_tip_focus)

        Ex_noise = e_lat_noise_vnm * (np.random.randn())

        n_evt = 0
        total_rate = 0.0

        # (1) injection: simplified
        if E_avg_vnm > 0.0:
            z = 1
            for x in range(w):
                if occ[x, z] == S_EMPTY:
                    Eb_inj = phib_bottom + e_inject0 - 0.15 * E_mig
                    Eb_inj = _clamp(Eb_inj, min_barrier)
                    r = NU0 * np.exp(-Eb_inj / (KB * T))
                    rates[n_evt] = r; etype[n_evt] = 3
                    ex[n_evt] = x; ez[n_evt] = z
                    enx[n_evt] = x; enz[n_evt] = z
                    total_rate += r; n_evt += 1
        elif E_avg_vnm < 0.0:
            z = h - 2
            for x in range(w):
                if occ[x, z] == S_EMPTY:
                    Eb_inj = phib_top + e_inject0 - 0.15 * E_mig
                    Eb_inj = _clamp(Eb_inj, min_barrier)
                    r = NU0 * np.exp(-Eb_inj / (KB * T))
                    rates[n_evt] = r; etype[n_evt] = 4
                    ex[n_evt] = x; ez[n_evt] = z
                    enx[n_evt] = x; enz[n_evt] = z
                    total_rate += r; n_evt += 1

        # (2) iterate oxide sites
        for z in range(1, h-1):
            for x in range(w):
                s = occ[x, z]

                if s == S_ION:
                    # z hop (use em_z)
                    if z+1 <= h-2 and occ[x, z+1] == S_EMPTY:
                        Eb = em_z[x, z] - alpha_field * E_mig * 1.0
                        Eb = _clamp(Eb, min_barrier)
                        r = NU0 * np.exp(-Eb / (KB * T))
                        if z >= tip_z and z <= tip_z + tip_ahead:
                            na = _count_attach_neighbors_including_electrodes(occ, x, z)
                            if na > 0: r *= hop_slow_near_tip
                        rates[n_evt] = r; etype[n_evt] = 0
                        ex[n_evt] = x; ez[n_evt] = z
                        enx[n_evt] = x; enz[n_evt] = z+1
                        total_rate += r; n_evt += 1

                    if z-1 >= 1 and occ[x, z-1] == S_EMPTY:
                        Eb = em_z[x, z] - alpha_field * E_mig * (-1.0)
                        Eb = _clamp(Eb, min_barrier)
                        r = NU0 * np.exp(-Eb / (KB * T))
                        if z >= tip_z and z <= tip_z + tip_ahead:
                            na = _count_attach_neighbors_including_electrodes(occ, x, z)
                            if na > 0: r *= hop_slow_near_tip
                        rates[n_evt] = r; etype[n_evt] = 0
                        ex[n_evt] = x; ez[n_evt] = z
                        enx[n_evt] = x; enz[n_evt] = z-1
                        total_rate += r; n_evt += 1

                    # x hop (use em_x)
                    if x-1 >= 0 and occ[x-1, z] == S_EMPTY:
                        Eb = em_x[x, z] - alpha_field * (Ex_noise) * (-1.0)
                        Eb = _clamp(Eb, min_barrier)
                        r = NU0 * np.exp(-Eb / (KB * T))
                        if z >= tip_z and z <= tip_z + tip_ahead:
                            na = _count_attach_neighbors_including_electrodes(occ, x, z)
                            if na > 0: r *= hop_slow_near_tip
                        rates[n_evt] = r; etype[n_evt] = 0
                        ex[n_evt] = x; ez[n_evt] = z
                        enx[n_evt] = x-1; enz[n_evt] = z
                        total_rate += r; n_evt += 1

                    if x+1 <= w-1 and occ[x+1, z] == S_EMPTY:
                        Eb = em_x[x, z] - alpha_field * (Ex_noise) * (1.0)
                        Eb = _clamp(Eb, min_barrier)
                        r = NU0 * np.exp(-Eb / (KB * T))
                        if z >= tip_z and z <= tip_z + tip_ahead:
                            na = _count_attach_neighbors_including_electrodes(occ, x, z)
                            if na > 0: r *= hop_slow_near_tip
                        rates[n_evt] = r; etype[n_evt] = 0
                        ex[n_evt] = x; ez[n_evt] = z
                        enx[n_evt] = x+1; enz[n_evt] = z
                        total_rate += r; n_evt += 1

                    # growth: ION -> FIL
                    if do_growth:
                        if z >= tip_z - tip_behind and z <= tip_z + tip_ahead:
                            na_fil = _count_fil_neighbors_only(occ, x, z)
                            if na_fil > 0:
                                Eb_g = e_grow0 - gamma_grow * E_tip - delta_attach * na_fil
                                Eb_g = _clamp(Eb_g, min_barrier)
                                r = NU0 * np.exp(-Eb_g / (KB * T))

                                l, rr, d, u = _neighbor_flags_fil(occ, x, z)
                                orient = 1.0
                                if d and (not l) and (not rr):
                                    orient = grow_pref_forward
                                elif (l or rr) and (not d):
                                    orient = grow_penalty_lateral
                                r *= orient

                                if focus_kappa > 0.0:
                                    dx = abs(x - tip_x)
                                    r *= 1.0 / (1.0 + focus_kappa * dx * dx)

                                r *= _center_bias_factor(x, w, enable_center_bias,
                                                        center_bias_strength, center_bias_sigma_cells)

                                rates[n_evt] = r; etype[n_evt] = 1
                                ex[n_evt] = x; ez[n_evt] = z
                                enx[n_evt] = x; enz[n_evt] = z
                                total_rate += r; n_evt += 1

                elif s == S_FIL and do_dissolve:
                    # FIL -> ION if has empty neighbor
                    na = 0
                    if x > 0   and occ[x-1, z] == S_EMPTY: na += 1
                    if x < w-1 and occ[x+1, z] == S_EMPTY: na += 1
                    if z > 1   and occ[x, z-1] == S_EMPTY: na += 1
                    if z < h-2 and occ[x, z+1] == S_EMPTY: na += 1
                    if na > 0:
                        Eb_d = e_diss0 - gamma_diss * E_tip
                        Eb_d = _clamp(Eb_d, min_barrier)
                        r = NU0 * np.exp(-Eb_d / (KB * T))
                        rates[n_evt] = r; etype[n_evt] = 2
                        ex[n_evt] = x; ez[n_evt] = z
                        enx[n_evt] = x; enz[n_evt] = z
                        total_rate += r; n_evt += 1

        if total_rate < 1e-40 or n_evt <= 0:
            energy_proxy_ev = abs(E_avg_vnm) * cell_z_nm * n_hopz
            return (occ, t_connect, connected, rates_s, ns, last_diss_x, last_diss_z,
                    n_hopx, n_hopz, n_grow, n_diss, n_inj, energy_proxy_ev)

        # KMC time increment
        u = np.random.random()
        dt = -np.log(u) / total_rate
        t += dt

        # roulette select
        rsel = np.random.random() * total_rate
        csum = 0.0
        sel = -1
        for i in range(n_evt):
            csum += rates[i]
            if csum >= rsel:
                sel = i
                break
        if sel < 0:
            continue

        if (step % rate_sample_stride == 0) and (ns < rate_sample_max):
            rates_s[ns] = rates[sel]
            ns += 1

        tp = etype[sel]
        x0 = ex[sel]; z0 = ez[sel]
        x1 = enx[sel]; z1 = enz[sel]

        if tp == 0:
            occ[x0, z0] = S_EMPTY
            occ[x1, z1] = S_ION
            if x1 != x0: n_hopx += 1
            if z1 != z0: n_hopz += 1
        elif tp == 1:
            occ[x0, z0] = S_FIL
            n_grow += 1
        elif tp == 2:
            occ[x0, z0] = S_ION
            last_diss_x = x0; last_diss_z = z0
            n_diss += 1
        elif tp == 3:
            occ[x0, z0] = S_ION
            n_inj += 1
        elif tp == 4:
            occ[x0, z0] = S_ION
            n_inj += 1

    mask = _bfs_bottom_connected_mask(occ)
    connected = _connected_to_top(occ, mask)
    energy_proxy_ev = abs(E_avg_vnm) * cell_z_nm * n_hopz
    return (occ, t_connect, connected, rates_s, ns, last_diss_x, last_diss_z,
            n_hopx, n_hopz, n_grow, n_diss, n_inj, energy_proxy_ev)


# =========================
# 8) wrappers: SET / LRS / RESET
# =========================
def run_set(occ0, em_x, em_z, E_vnm, T, mc_seed):
    return _run_kmc(
        occ0, em_x, em_z,
        E_vnm, T,
        MAX_STEPS_SET,
        True, False,
        STOP_IF_CONNECTED_SET, False,
        mc_seed,
        RATE_SAMPLE_MAX, RATE_SAMPLE_STRIDE,
        MASK_UPDATE_STRIDE,
        ALPHA_FIELD, MIN_BARRIER_EV,
        FIELD_ENHANCE_MIG, FIELD_ENHANCE_GROW,
        TIP_FOCUS_POWER, TIP_FOCUS_MAX_VNM, ENABLE_TIP_FOCUS,
        E_GROW0_EV, GAMMA_GROW, DELTA_ATTACH,
        HOP_SLOW_NEAR_TIP, TIP_BEHIND, TIP_AHEAD,
        E_DISS0_EV, GAMMA_DISS,
        PHIB_BOTTOM, PHIB_TOP, E_INJECT0_EV,
        FOCUS_KAPPA,
        E_LATERAL_NOISE_VNM,
        GROW_PREF_FORWARD,
        GROW_PENALTY_LATERAL,
        CELL_Z_NM,
        ENABLE_CENTER_BIAS,
        CENTER_BIAS_STRENGTH,
        CENTER_BIAS_SIGMA_CELLS
    )

def run_post_thicken(occ_crit, em_x, em_z, E_vnm, T, mc_seed):
    return _run_kmc(
        occ_crit, em_x, em_z,
        E_vnm, T,
        POST_CONNECT_STEPS,
        True, False,
        False, False,
        mc_seed,
        RATE_SAMPLE_MAX, RATE_SAMPLE_STRIDE,
        MASK_UPDATE_STRIDE,
        ALPHA_FIELD, MIN_BARRIER_EV,
        FIELD_ENHANCE_MIG, FIELD_ENHANCE_GROW,
        TIP_FOCUS_POWER, TIP_FOCUS_MAX_VNM, ENABLE_TIP_FOCUS,
        E_GROW0_EV, GAMMA_GROW, DELTA_ATTACH,
        HOP_SLOW_NEAR_TIP, TIP_BEHIND, TIP_AHEAD,
        E_DISS0_EV, GAMMA_DISS,
        PHIB_BOTTOM, PHIB_TOP, E_INJECT0_EV,
        FOCUS_KAPPA,
        E_LATERAL_NOISE_VNM,
        GROW_PREF_FORWARD,
        GROW_PENALTY_LATERAL,
        CELL_Z_NM,
        ENABLE_CENTER_BIAS,
        CENTER_BIAS_STRENGTH,
        CENTER_BIAS_SIGMA_CELLS
    )

def run_reset(occ_lrs, em_x, em_z, E_vnm, T, mc_seed):
    return _run_kmc(
        occ_lrs, em_x, em_z,
        -abs(E_vnm), T,
        MAX_STEPS_RESET,
        False, True,
        False, STOP_IF_DISCONNECTED_RESET,
        mc_seed,
        RATE_SAMPLE_MAX, RATE_SAMPLE_STRIDE,
        MASK_UPDATE_STRIDE,
        ALPHA_FIELD, MIN_BARRIER_EV,
        FIELD_ENHANCE_MIG, FIELD_ENHANCE_GROW,
        TIP_FOCUS_POWER, TIP_FOCUS_MAX_VNM, ENABLE_TIP_FOCUS,
        E_GROW0_EV, GAMMA_GROW, DELTA_ATTACH,
        HOP_SLOW_NEAR_TIP, TIP_BEHIND, TIP_AHEAD,
        E_DISS0_EV, GAMMA_DISS,
        PHIB_BOTTOM, PHIB_TOP, E_INJECT0_EV,
        FOCUS_KAPPA,
        E_LATERAL_NOISE_VNM,
        GROW_PREF_FORWARD,
        GROW_PENALTY_LATERAL,
        CELL_Z_NM,
        ENABLE_CENTER_BIAS,
        CENTER_BIAS_STRENGTH,
        CENTER_BIAS_SIGMA_CELLS
    )


# =========================
# 9) one simulation (device_seed vs mc_seed separation)
# =========================
def morphology_from_occ(occ):
    mask = _bfs_bottom_connected_mask(occ)
    return _morphology_metrics_filament(occ, mask, CELL_X_NM, DRIFT_THRESH_NM)

def simulate_one(E_vnm, T, device_seed, mc_seed, barrier_mode="random",
                 store_rate_samples=False, return_snapshots=False):
    # barrier(s)
    if barrier_mode == "random":
        em = make_barrier_map_random(seed=device_seed)
        em_x, em_z = em, em
    elif barrier_mode == "layered":
        z_regions = [(0, 0), (1, 10), (11, 24), (25, THICKNESS_CELLS-1)]
        Eb_values = [BASE_BARRIER_EV, 0.78, 0.90, BASE_BARRIER_EV]
        em = make_barrier_map_layered(z_regions, Eb_values, sigma=RANDOM_SIGMA_EV, seed=device_seed)
        em_x, em_z = em, em
    else:
        em_x, em_z = make_barrier_maps_aniso(Eb_x=0.95, Eb_z=0.75, sigma=RANDOM_SIGMA_EV, seed=device_seed)

    occ0 = init_occ(mc_seed)
    snaps = {"HRS0": occ0.copy()} if return_snapshots else None

    occ_crit, t_set, connected, rates_s, ns, _, _, n_hopx, n_hopz, n_grow, n_diss, n_inj, energy_set = run_set(
        occ0, em_x, em_z, E_vnm, T, mc_seed
    )
    formed = bool(connected)
    if return_snapshots:
        snaps["Critical"] = occ_crit.copy()

    # LRS
    occ_lrs = occ_crit
    if formed and ENABLE_POST_THICKEN:
        occ_lrs, *_ = run_post_thicken(occ_crit, em_x, em_z, E_vnm, T, mc_seed + 77777)
        occ_lrs = occ_lrs
    if return_snapshots:
        snaps["LRS"] = occ_lrs.copy()

    # RESET
    switched = False
    t_reset = np.nan
    rupture_x_nm = np.nan
    rupture_z_nm = np.nan
    energy_reset = np.nan

    if ENABLE_RESET and (not FAST_MODE) and formed:
        occ_res, t_conn2, connected2, _rs, _ns, lastx, lastz, *_stats = run_reset(occ_lrs, em_x, em_z, E_vnm, T, mc_seed + 100000)
        switched = (not connected2)
        t_reset = t_conn2
        if lastx >= 0:
            rupture_x_nm = lastx * CELL_X_NM
            rupture_z_nm = lastz * CELL_Z_NM
        energy_reset = _stats[-1] if len(_stats) else np.nan
        if return_snapshots:
            snaps["HRS_reset"] = occ_res.copy()

    meander_rms_nm, neck_nm, branches, comx_nm, drift_flag, tort_nm = morphology_from_occ(occ_lrs)

    out = dict(
        device_seed=int(device_seed),
        mc_seed=int(mc_seed),
        E=float(E_vnm),
        T=float(T),
        formed=int(formed),
        switched=int(switched),
        phase=2 if (formed and switched) else (1 if formed else 0),
        t_set=float(t_set),
        t_reset=float(t_reset),
        energy_set_ev=float(energy_set),
        energy_reset_ev=float(energy_reset) if np.isfinite(energy_reset) else np.nan,
        meander_rms_nm=float(meander_rms_nm),
        tortuosity_nm=float(tort_nm),
        neck_nm=float(neck_nm),
        branches=int(branches),
        comx_nm=float(comx_nm),
        drift=int(drift_flag),
        rupture_x_nm=float(rupture_x_nm),
        rupture_z_nm=float(rupture_z_nm),
        rate_var_log=float(np.var(np.log10(rates_s[:ns] + 1e-40))) if ns > 5 else np.nan,
    )

    if store_rate_samples:
        out["rate_samples"] = rates_s[:max(1, ns)].copy()

    if return_snapshots:
        out["snapshots"] = snaps
        out["em_x"] = em_x
        out["em_z"] = em_z

    return out


# =========================
# 10) plotting: barrier + states + 3D revolve
# =========================
def plot_barrier_map(em, title="Barrier map (eV)"):
    fig = plt.figure(figsize=(6.6, 4.0))
    ax = fig.add_subplot(1,1,1)
    im = ax.imshow(em[:, 1:-1].T, origin="lower", aspect="auto",
                   extent=[0, XSPAN_NM, 0, TOX_NM])
    ax.set_title(title)
    ax.set_xlabel("x (nm)")
    ax.set_ylabel("z (nm)")
    ax.grid(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Eb (eV)")
    plt.tight_layout()
    plt.show()

    col_mean = em.mean(axis=1)
    fig = plt.figure(figsize=(6.6, 2.8))
    ax = fig.add_subplot(1,1,1)
    xs = np.arange(WIDTH_CELLS) * CELL_X_NM
    ax.plot(xs, col_mean, marker="o", lw=1.5)
    ax.set_title("Column-mean Eb vs x (diagnose left/right bias)")
    ax.set_xlabel("x (nm)")
    ax.set_ylabel("mean Eb (eV)")
    _no_offset(ax)
    plt.tight_layout()
    plt.show()

def _plot_occ_states(ax, occ, title):
    occ_ox = occ[:, 1:-1].T
    cmap = ListedColormap(["white", "#808080", "#cc2b5e"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)
    im = ax.imshow(occ_ox, origin="lower", aspect="auto", cmap=cmap, norm=norm,
                   extent=[0, XSPAN_NM, 0, TOX_NM])
    ax.set_title(title)
    ax.set_xlabel("x (nm)")
    ax.set_ylabel("z (nm)")
    ax.grid(False)
    return im

def _plot_mask(ax, occ, title="Bottom-connected FIL mask"):
    mask = _bfs_bottom_connected_mask(occ)
    fil = ((mask[:, 1:-1] == 1) & (occ[:, 1:-1] == S_FIL)).T
    ax.imshow(fil.astype(np.uint8), origin="lower", aspect="auto", cmap="Blues",
              extent=[0, XSPAN_NM, 0, TOX_NM])
    ax.set_title(title)
    ax.set_xlabel("x (nm)")
    ax.set_ylabel("z (nm)")
    ax.grid(False)

def _plot_ion_map(ax, occ, title="ION distribution"):
    ion = (occ[:, 1:-1] == S_ION).T
    ax.imshow(ion.astype(np.uint8), origin="lower", aspect="auto", cmap="Greys",
              extent=[0, XSPAN_NM, 0, TOX_NM])
    ax.set_title(title)
    ax.set_xlabel("x (nm)")
    ax.set_ylabel("z (nm)")
    ax.grid(False)

def filament_profile(mask, occ):
    w, h = occ.shape
    zs, widths, comxs = [], [], []
    for z in range(1, h-1):
        xs = []
        for x in range(w):
            if mask[x, z] == 1 and occ[x, z] == S_FIL:
                xs.append(x)
        if len(xs) > 0:
            zs.append(z)
            widths.append(len(xs))
            comxs.append(float(np.mean(xs)))
    if len(zs) == 0:
        return np.array([]), np.array([]), np.array([])
    return np.array(zs), np.array(widths), np.array(comxs)

def plot_revolved_filament(ax, occ):
    mask = _bfs_bottom_connected_mask(occ)
    zs, widths, comxs = filament_profile(mask, occ)
    if len(zs) < 3:
        ax.text2D(0.1, 0.5, "No filament profile", transform=ax.transAxes)
        return
    z_nm = (zs - Z_OX_START) * CELL_Z_NM
    r_nm = np.maximum(widths * CELL_X_NM * 0.5, 0.4 * CELL_X_NM)
    center0 = (WIDTH_CELLS - 1) / 2.0
    x0_nm = (comxs - center0) * CELL_X_NM

    theta = np.linspace(0, 2*np.pi, 56)
    Zm, Th = np.meshgrid(z_nm, theta)
    x0_grid = np.tile(x0_nm, (len(theta), 1))
    r_grid = np.tile(r_nm, (len(theta), 1))
    X = x0_grid + r_grid * np.cos(Th)
    Y = r_grid * np.sin(Th)
    Z = Zm
    ax.plot_surface(X, Y, Z, linewidth=0, antialiased=True, alpha=0.78)
    ax.set_xlabel("x (nm, center-shift)")
    ax.set_ylabel("y (nm)")
    ax.set_zlabel("z (nm)")

def plot_demo_four_states(demo_out):
    snaps = demo_out["snapshots"]
    states = ["HRS0", "Critical", "LRS"]
    if "HRS_reset" in snaps:
        states.append("HRS_reset")

    ncol = len(states)
    fig = plt.figure(figsize=(18, 9))
    for i, st in enumerate(states):
        occ = snaps[st]
        ax1 = fig.add_subplot(3, ncol, 1+i)
        im = _plot_occ_states(ax1, occ, f"{st}: states")
        if i == ncol - 1:
            cb = fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
            cb.set_ticks([0,1,2])
            cb.set_ticklabels(["EMPTY","ION","FIL"])

        ax2 = fig.add_subplot(3, ncol, 1+ncol+i)
        _plot_mask(ax2, occ, f"{st}: connected FIL")

        ax3 = fig.add_subplot(3, ncol, 1+2*ncol+i, projection="3d")
        plot_revolved_filament(ax3, occ)
        ax3.set_title(f"{st}: 3D revolve")
        ax3.view_init(elev=18, azim=35)

    plt.tight_layout()
    plt.show()

    fig = plt.figure(figsize=(18, 3.8))
    for i, st in enumerate(states):
        ax = fig.add_subplot(1, ncol, 1+i)
        _plot_ion_map(ax, snaps[st], f"{st}: ION map")
    plt.tight_layout()
    plt.show()

    rows = []
    for st in states:
        occ = snaps[st]
        mask = _bfs_bottom_connected_mask(occ)
        meander, neck, br, comx, drift, tort = _morphology_metrics_filament(occ, mask, CELL_X_NM, DRIFT_THRESH_NM)
        rows.append([st, meander, neck, br, comx, drift, tort])
    dfm = pd.DataFrame(rows, columns=["state","meander_rms_nm","neck_nm","branches","comx_nm","drift","tortuosity_nm"])
    print("\n[DEMO] Morphology table:\n", dfm.to_string(index=False))


# =========================
# 11) Fig7-like plots (phase map, tset heatmap, fixed boxplots, morphology vs E)
# =========================
def plot_fig7(df):
    phase_mat = np.zeros((len(E_VALS), len(T_VALS)), dtype=int)
    for i, E in enumerate(E_VALS):
        for j, T in enumerate(T_VALS):
            sub = df[(df["E"] == float(E)) & (df["T"] == float(T))]
            pf = sub["formed"].mean() if len(sub) else 0.0
            ps = sub["switched"].mean() if len(sub) else 0.0
            if pf < P_FORM_MIN:
                phase_mat[i, j] = 0
            else:
                phase_mat[i, j] = 2 if (ps >= P_SWITCH_MIN) else 1

    tmean = np.full((len(E_VALS), len(T_VALS)), np.nan, dtype=float)
    for i, E in enumerate(E_VALS):
        for j, T in enumerate(T_VALS):
            sub = df[(df["E"] == float(E)) & (df["T"] == float(T)) & (df["formed"] == 1)]
            if len(sub) >= 3:
                vals = sub["t_set"].values
                vals = vals[np.isfinite(vals)]
                if len(vals) >= 3:
                    tmean[i, j] = np.log10(np.mean(vals) + 1e-40)

    fig = plt.figure(figsize=(18, 10))
    ax1 = fig.add_subplot(2, 2, 1)
    cmap_phase = ListedColormap(["#2b2b2b", "#f0c419", "#2ecc71"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap_phase.N)
    sns.heatmap(
        phase_mat, ax=ax1, cmap=cmap_phase, norm=norm, cbar=True,
        xticklabels=[f"{int(round(t))}" for t in T_VALS],
        yticklabels=[f"{e:.3f}" for e in E_VALS],
    )
    ax1.invert_yaxis()
    ax1.set_title("(a) E–T Phase Map (x=T[K], y=E[V/nm])  0/1/2")
    ax1.set_xlabel("T (K)")
    ax1.set_ylabel("E (V/nm)")

    ax2 = fig.add_subplot(2, 2, 2)
    sns.heatmap(
        tmean, ax=ax2, cmap="magma", cbar=True,
        xticklabels=[f"{int(round(t))}" for t in T_VALS],
        yticklabels=[f"{e:.3f}" for e in E_VALS],
    )
    ax2.invert_yaxis()
    ax2.set_title("(b) log10(mean t_SET)  (NaN where not forming)")
    ax2.set_xlabel("T (K)")
    ax2.set_ylabel("E (V/nm)")

    df_ok = df[(df["formed"] == 1) & np.isfinite(df["t_set"])].copy()
    ax3 = fig.add_subplot(2, 2, 3)
    if len(df_ok) > 10:
        df_ok["log_tset"] = np.log10(df_ok["t_set"] + 1e-40)
        orderE = [f"{e:.3f}" for e in E_VALS]
        orderT = [f"{int(round(t))}" for t in T_VALS]
        df_ok["E_str"] = df_ok["E"].map(lambda v: f"{v:.3f}")
        df_ok["T_str"] = df_ok["T"].map(lambda v: f"{int(round(v))}")
        Ts = np.quantile(T_VALS, [0.2, 0.5, 0.8])
        Ts = [float(T_VALS[np.argmin(abs(T_VALS - t))]) for t in Ts]
        sub = df_ok[df_ok["T"].isin(Ts)].copy()
        sub["E_str"] = pd.Categorical(sub["E_str"], categories=orderE, ordered=True)
        sub["T_str"] = pd.Categorical(sub["T_str"], categories=orderT, ordered=True)
        sns.boxplot(data=sub, x="E_str", y="log_tset", hue="T_str",
                    ax=ax3, order=orderE, fliersize=2.5, linewidth=0.9)
        ax3.set_title("(b-1) log10(t_SET) vs E  (3 T slices)")
        ax3.set_xlabel("E (V/nm)")
        ax3.set_ylabel("log10(t_SET / s)")
        ax3.tick_params(axis="x", rotation=45)
        ax3.legend(title="T (K)", loc="best", ncol=1)
        _no_offset(ax3)
    else:
        ax3.text(0.5, 0.5, "Not enough forming samples", ha="center", va="center")

    ax4 = fig.add_subplot(2, 2, 4)
    if len(df_ok) > 10:
        df_ok["E_str"] = df_ok["E"].map(lambda v: f"{v:.3f}")
        df_ok["T_str"] = df_ok["T"].map(lambda v: f"{int(round(v))}")
        Es = np.quantile(E_VALS, [0.2, 0.5, 0.8])
        Es = [float(E_VALS[np.argmin(abs(E_VALS - e))]) for e in Es]
        sub = df_ok[df_ok["E"].isin(Es)].copy()
        orderT = [f"{int(round(t))}" for t in T_VALS]
        orderE = [f"{e:.3f}" for e in E_VALS]
        sub["T_str"] = pd.Categorical(sub["T_str"], categories=orderT, ordered=True)
        sub["E_str"] = pd.Categorical(sub["E_str"], categories=orderE, ordered=True)
        sns.boxplot(data=sub, x="T_str", y="log_tset", hue="E_str",
                    ax=ax4, order=orderT, fliersize=2.5, linewidth=0.9)
        ax4.set_title("(b-2) log10(t_SET) vs T  (3 E slices)")
        ax4.set_xlabel("T (K)")
        ax4.set_ylabel("log10(t_SET / s)")
        ax4.tick_params(axis="x", rotation=45)
        ax4.legend(title="E (V/nm)", loc="best", ncol=1)
        _no_offset(ax4)
    else:
        ax4.text(0.5, 0.5, "Not enough forming samples", ha="center", va="center")

    plt.tight_layout()
    plt.show()

    # morphology vs E @ T_ref
    subm = df[(df["T"] == float(T_REF_MORPH)) & (df["formed"] == 1)]
    if len(subm) > 0:
        g = subm.groupby("E")
        E_line = np.array(sorted(g.groups.keys()))
        M = np.array([g.get_group(e)["meander_rms_nm"].mean() for e in E_line])
        Ms = np.array([g.get_group(e)["meander_rms_nm"].std() for e in E_line])
        Nn = np.array([g.get_group(e)["neck_nm"].mean() for e in E_line])
        Ns = np.array([g.get_group(e)["neck_nm"].std() for e in E_line])
        B = np.array([g.get_group(e)["branches"].mean() for e in E_line])
        Bs = np.array([g.get_group(e)["branches"].std() for e in E_line])
        D = np.array([g.get_group(e)["drift"].mean() for e in E_line])

        fig2 = plt.figure(figsize=(16, 4))
        ax = fig2.add_subplot(1, 4, 1)
        ax.errorbar(E_line, M, yerr=Ms, marker="o", lw=1.3, capsize=3)
        ax.set_title("Meander RMS (nm)")
        ax.set_xlabel("E (V/nm)"); ax.set_ylabel("nm"); _no_offset(ax)

        ax = fig2.add_subplot(1, 4, 2)
        ax.errorbar(E_line, Nn, yerr=Ns, marker="o", lw=1.3, capsize=3)
        ax.set_title("Min neck (nm)")
        ax.set_xlabel("E (V/nm)"); ax.set_ylabel("nm"); _no_offset(ax)

        ax = fig2.add_subplot(1, 4, 3)
        ax.errorbar(E_line, B, yerr=Bs, marker="o", lw=1.3, capsize=3)
        ax.set_title("Branch count")
        ax.set_xlabel("E (V/nm)"); ax.set_ylabel("count"); _no_offset(ax)

        ax = fig2.add_subplot(1, 4, 4)
        ax.plot(E_line, D, marker="o", lw=1.3)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title("Drift probability")
        ax.set_xlabel("E (V/nm)"); ax.set_ylabel("P(drift)"); _no_offset(ax)

        fig2.suptitle(f"Fig.7(c) morphology @ T={T_REF_MORPH:.0f} K (mean±std)")
        plt.tight_layout()
        plt.show()


# =========================
# 12) Convergence study (multi-T)
# =========================
def convergence_study(E0, T_list, N_list, device_seed=2025, barrier_mode="random"):
    rows_all = []
    for T0 in T_list:
        rows = []
        for N in N_list:
            outs = []
            for k in range(N):
                out = simulate_one(E0, T0, device_seed=device_seed, mc_seed=1000+k,
                                   barrier_mode=barrier_mode, store_rate_samples=False, return_snapshots=False)
                outs.append(out)
            df = pd.DataFrame(outs)
            p_form = df["formed"].mean()
            df_f = df[(df["formed"] == 1) & np.isfinite(df["t_set"])].copy()
            if len(df_f) > 0:
                logt = np.log10(df_f["t_set"].values + 1e-40)
                mean_logt = float(np.mean(logt))
                std_logt = float(np.std(logt))
                mean_energy = float(np.mean(df_f["energy_set_ev"].values))
                std_energy = float(np.std(df_f["energy_set_ev"].values))
            else:
                mean_logt = np.nan; std_logt = np.nan
                mean_energy = np.nan; std_energy = np.nan
            rows.append([T0, N, mean_logt, std_logt, mean_energy, std_energy, p_form])
            rows_all.append([T0, N, mean_logt, std_logt, mean_energy, std_energy, p_form])

        dfT = pd.DataFrame(rows, columns=["T","N","mean_logt","std_logt","mean_energy_ev","std_energy_ev","p_form"])
        print(f"\n>>> Convergence study @ E={E0:.3f} V/nm, T={T0:.0f} K (device_seed={device_seed})")
        print(dfT.to_string(index=False))

        fig = plt.figure(figsize=(14.2, 3.6))
        ax1 = fig.add_subplot(1,3,1)
        ax1.plot(dfT["N"], dfT["p_form"], marker="o", lw=1.5)
        ax1.set_ylim(-0.02, 1.02)
        ax1.set_title("Forming probability")
        ax1.set_xlabel("N seeds"); ax1.set_ylabel("P(form)"); _no_offset(ax1)

        ax2 = fig.add_subplot(1,3,2)
        ax2.plot(dfT["N"], dfT["mean_logt"], marker="o", lw=1.5, label="mean")
        ax2.fill_between(dfT["N"], dfT["mean_logt"]-dfT["std_logt"], dfT["mean_logt"]+dfT["std_logt"], alpha=0.15, label="±1 std")
        ax2.set_title("t_SET stability")
        ax2.set_xlabel("N seeds"); ax2.set_ylabel("mean±std  log10(t_SET / s)")
        ax2.legend(loc="best", frameon=True); _no_offset(ax2)

        ax3 = fig.add_subplot(1,3,3)
        ax3.plot(dfT["N"], dfT["mean_energy_ev"], marker="o", lw=1.5, label="mean")
        ax3.fill_between(dfT["N"], dfT["mean_energy_ev"]-dfT["std_energy_ev"], dfT["mean_energy_ev"]+dfT["std_energy_ev"], alpha=0.15, label="±1 std")
        ax3.set_title("Energy proxy stability")
        ax3.set_xlabel("N seeds"); ax3.set_ylabel("mean±std  energy proxy (eV)")
        ax3.legend(loc="best", frameon=True); _no_offset(ax3)

        fig.suptitle(f"Convergence @ E={E0:.3f} V/nm, T={T0:.0f} K")
        plt.tight_layout()
        plt.show()

    return pd.DataFrame(rows_all, columns=["T","N","mean_logt","std_logt","mean_energy_ev","std_energy_ev","p_form"])


# =========================
# 13) Sensitivity (local ±pert)
# =========================
def sensitivity_analysis(E0, T0, device_seed=2025, nseeds=16, pert_frac=0.15):
    base = dict(
        PHIB_BOTTOM=PHIB_BOTTOM,
        PHIB_TOP=PHIB_TOP,
        E_INJECT0_EV=E_INJECT0_EV,
        FIELD_ENHANCE_MIG=FIELD_ENHANCE_MIG,
        FIELD_ENHANCE_GROW=FIELD_ENHANCE_GROW,
        E_GROW0_EV=E_GROW0_EV,
        GAMMA_GROW=GAMMA_GROW,
        DELTA_ATTACH=DELTA_ATTACH,
        E_DISS0_EV=E_DISS0_EV,
        GAMMA_DISS=GAMMA_DISS,
        FOCUS_KAPPA=FOCUS_KAPPA,
        E_LATERAL_NOISE_VNM=E_LATERAL_NOISE_VNM,
        CENTER_BIAS_STRENGTH=CENTER_BIAS_STRENGTH,
    )

    def run_with(over):
        em = make_barrier_map_random(seed=device_seed)
        em_x = em; em_z = em

        phib_b = over.get("PHIB_BOTTOM", PHIB_BOTTOM)
        phib_t = over.get("PHIB_TOP", PHIB_TOP)
        e_inj0 = over.get("E_INJECT0_EV", E_INJECT0_EV)
        f_mig  = over.get("FIELD_ENHANCE_MIG", FIELD_ENHANCE_MIG)
        f_grow = over.get("FIELD_ENHANCE_GROW", FIELD_ENHANCE_GROW)
        e_g0   = over.get("E_GROW0_EV", E_GROW0_EV)
        gg     = over.get("GAMMA_GROW", GAMMA_GROW)
        da     = over.get("DELTA_ATTACH", DELTA_ATTACH)
        ed0    = over.get("E_DISS0_EV", E_DISS0_EV)
        gd     = over.get("GAMMA_DISS", GAMMA_DISS)
        fk     = over.get("FOCUS_KAPPA", FOCUS_KAPPA)
        eln    = over.get("E_LATERAL_NOISE_VNM", E_LATERAL_NOISE_VNM)
        cbs    = over.get("CENTER_BIAS_STRENGTH", CENTER_BIAS_STRENGTH)

        outs = []
        for k in range(nseeds):
            mc_seed = 2000 + k
            occ0 = init_occ(mc_seed)
            occ, t_set, conn, *_rest = _run_kmc(
                occ0, em_x, em_z,
                E0, T0,
                MAX_STEPS_SET,
                True, False,
                True, False,
                mc_seed,
                RATE_SAMPLE_MAX, RATE_SAMPLE_STRIDE,
                MASK_UPDATE_STRIDE,
                ALPHA_FIELD, MIN_BARRIER_EV,
                f_mig, f_grow,
                TIP_FOCUS_POWER, TIP_FOCUS_MAX_VNM, ENABLE_TIP_FOCUS,
                e_g0, gg, da,
                HOP_SLOW_NEAR_TIP, TIP_BEHIND, TIP_AHEAD,
                ed0, gd,
                phib_b, phib_t, e_inj0,
                fk,
                eln,
                GROW_PREF_FORWARD,
                GROW_PENALTY_LATERAL,
                CELL_Z_NM,
                ENABLE_CENTER_BIAS,
                cbs,
                CENTER_BIAS_SIGMA_CELLS
            )
            formed = int(conn)
            energy = _rest[-1]
            outs.append(dict(formed=formed, t_set=float(t_set), energy=float(energy)))
        df = pd.DataFrame(outs)
        df_f = df[(df["formed"]==1) & np.isfinite(df["t_set"])]
        if len(df_f) == 0:
            return dict(mean_logt=np.nan, std_logt=np.nan, mean_energy=np.nan, p_form=float(df["formed"].mean()))
        logt = np.log10(df_f["t_set"].values + 1e-40)
        return dict(
            mean_logt=float(np.mean(logt)),
            std_logt=float(np.std(logt)),
            mean_energy=float(np.mean(df_f["energy"].values)),
            p_form=float(df["formed"].mean())
        )

    base_metrics = run_with({})
    rec = []
    for k, v in base.items():
        if abs(v) < 1e-12:
            continue
        mup = run_with({k: v*(1+pert_frac)})
        mdn = run_with({k: v*(1-pert_frac)})
        dmean_logt = (mup["mean_logt"] - mdn["mean_logt"]) / (2*pert_frac)
        dstd_logt  = (mup["std_logt"]  - mdn["std_logt"])  / (2*pert_frac)
        denergy    = (mup["mean_energy"] - mdn["mean_energy"]) / (2*pert_frac)
        dpform     = (mup["p_form"] - mdn["p_form"]) / (2*pert_frac)
        rec.append([k, dmean_logt, dstd_logt, denergy, dpform])

    dfs = pd.DataFrame(rec, columns=["param","d_mean_logt_per_frac","d_std_logt_per_frac","d_energy_per_frac","d_pform_per_frac"])
    dfs["abs_mean"] = np.abs(dfs["d_mean_logt_per_frac"])
    dfs = dfs.sort_values("abs_mean", ascending=True)

    fig = plt.figure(figsize=(14.5, 4.2))
    ax1 = fig.add_subplot(1,3,1)
    ax1.barh(dfs["param"], dfs["d_mean_logt_per_frac"])
    ax1.set_title("Sensitivity: mean log10(t_SET)")
    ax1.set_xlabel("d(metric)/d(frac)"); ax1.grid(True, axis="x", alpha=0.3)

    ax2 = fig.add_subplot(1,3,2)
    ax2.barh(dfs["param"], dfs["d_energy_per_frac"])
    ax2.set_title("Sensitivity: mean energy proxy")
    ax2.set_xlabel("d(metric)/d(frac)"); ax2.grid(True, axis="x", alpha=0.3)

    ax3 = fig.add_subplot(1,3,3)
    ax3.barh(dfs["param"], dfs["d_std_logt_per_frac"])
    ax3.set_title("Sensitivity: std log10(t_SET)")
    ax3.set_xlabel("d(metric)/d(frac)"); ax3.grid(True, axis="x", alpha=0.3)

    fig.suptitle(f"Sensitivity @ E={E0:.3f} V/nm, T={T0:.0f} K  (±{int(pert_frac*100)}%)")
    plt.tight_layout()
    plt.show()

    print("\n>>> Sensitivity table (sorted by |d mean_logt|):\n", dfs.drop(columns=["abs_mean"]).to_string(index=False))
    print("\n>>> Baseline metrics:", base_metrics)
    return dfs, base_metrics


# =========================
# 14) scan runner
# =========================
def run_scan():
    tasks = []
    for device_seed in DEVICE_SEEDS_SCAN:
        for E in E_VALS:
            for T in T_VALS:
                for s in range(N_SEEDS_PER_POINT):
                    tasks.append((device_seed, float(E), float(T), int(s)))
    print(f">>> Total sims = {len(tasks)}  (device_seeds={DEVICE_SEEDS_SCAN})", flush=True)

    def _task(device_seed, E, T, s):
        mc_seed = 10_000 * device_seed + 1000 * int(round(E*1000)) + 10 * int(round(T)) + s
        return simulate_one(E, T, device_seed=device_seed, mc_seed=mc_seed,
                            barrier_mode=BARRIER_MODE_SCAN,
                            store_rate_samples=STORE_RATE_SAMPLES_SCAN,
                            return_snapshots=False)

    results = []
    t_scan = time.time()
    for start in range(0, len(tasks), CHUNK_SIZE):
        chunk = tasks[start:start+CHUNK_SIZE]
        t1 = time.time()
        res = Parallel(
            n_jobs=N_JOBS,
            prefer=PARALLEL_PREFER,
            verbose=JOBLIB_VERBOSE,
            batch_size=1
        )(delayed(_task)(ds, E, T, s) for (ds, E, T, s) in chunk)
        results.extend(res)
        done = start + len(chunk)
        print(f">>> Progress: {done}/{len(tasks)} (+{len(chunk)} in {time.time()-t1:.1f}s, elapsed {time.time()-t_scan:.1f}s)", flush=True)
    return pd.DataFrame(results)


# =========================
# 15) main
# =========================
def main():
    t_all = time.time()
    try:
        sys.stdout.reconfigure(line_buffering=True)
    except Exception:
        pass

    print(f">>> FAST_MODE={FAST_MODE}", flush=True)
    print(f">>> tox={TOX_NM:.1f} nm, CELL_Z_NM={CELL_Z_NM:.3f} nm, CELL_X_NM={CELL_X_NM:.3f} nm", flush=True)
    print(f">>> Grid: W={WIDTH_CELLS}, H={THICKNESS_CELLS} (oxide z=1..{THICKNESS_CELLS-2})", flush=True)
    print(f">>> Scan: |E|={len(E_VALS)}, |T|={len(T_VALS)}, seeds/pt={N_SEEDS_PER_POINT}", flush=True)
    print(f">>> Barrier DEMO={BARRIER_MODE_DEMO}, SCAN={BARRIER_MODE_SCAN}", flush=True)
    print(f">>> Center bias growth={ENABLE_CENTER_BIAS} (strength={CENTER_BIAS_STRENGTH}, sigma={CENTER_BIAS_SIGMA_CELLS})", flush=True)

    # numba warmup
    print(">>> Numba warmup ...", flush=True)
    t0 = time.time()
    em_demo = make_barrier_map_random(seed=42)
    occ0 = init_occ(0)
    _ = run_set(occ0, em_demo, em_demo, E_DEMO, T_DEMO, mc_seed=0)
    print(f">>> Numba warmup done in {time.time()-t0:.2f}s", flush=True)

    # DEMO
    if RUN_DEMO:
        demo = simulate_one(E_DEMO, T_DEMO, device_seed=2025, mc_seed=1,
                            barrier_mode=BARRIER_MODE_DEMO,
                            store_rate_samples=STORE_RATE_SAMPLES_DEMO,
                            return_snapshots=True)
        print(f"[DEMO] formed={demo['formed']}, switched={demo['switched']}, phase={demo['phase']}, t_set={demo['t_set']}", flush=True)

        if BARRIER_MODE_DEMO == "aniso":
            plot_barrier_map(demo["em_z"], "Barrier map for z-hop (eV)")
            plot_barrier_map(demo["em_x"], "Barrier map for x-hop (eV)")
        else:
            plot_barrier_map(demo["em_z"], "Barrier map (eV)")

        plot_demo_four_states(demo)

        if "rate_samples" in demo:
            r = np.asarray(demo["rate_samples"])
            fig = plt.figure(figsize=(6.2, 3.4))
            ax = fig.add_subplot(1,1,1)
            sns.histplot(np.log10(r + 1e-40), bins=30, kde=True, ax=ax)
            ax.set_title("DEMO: log10(selected rate) distribution")
            ax.set_xlabel("log10(rate) [Hz]")
            _no_offset(ax)
            plt.tight_layout()
            plt.show()

    # SCAN
    df_scan = None
    if RUN_SCAN:
        print(">>> Running E–T scan ...", flush=True)
        df_scan = run_scan()
        print(">>> Scan finished. Plotting Fig.7 ...", flush=True)
        plot_fig7(df_scan)

    # convergence (multi-T)
    if RUN_CONVERGENCE:
        _ = convergence_study(CONV_E0, CONV_T_LIST, CONV_N_LIST, device_seed=2025, barrier_mode="random")

    # sensitivity
    if RUN_SENSITIVITY:
        _ = sensitivity_analysis(SENS_E0, SENS_T0, device_seed=2025, nseeds=SENS_NSEEDS, pert_frac=SENS_PERT_FRAC)

    print(f">>> ALL DONE in {time.time()-t_all:.1f}s", flush=True)
    return df_scan


if __name__ == "__main__":
    main()
